# 🔥 YOLOv8 Fine-Tuning Server
> Auto-generated worker for remote GPU training sessions.

**Runtime Setup:** Runtime → Change runtime type → **T4 GPU** → Save, then Run All.

In [ ]:
# ⚡ CELL 1 — Install & Download Setup (Run Once)
!pip install -q fastapi uvicorn python-multipart requests ultralytics pydantic

import os

# Download cloudflared binary if missing
if not os.path.exists('cloudflared-linux-amd64'):
    os.system('wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64')
    os.system('chmod +x cloudflared-linux-amd64')
    print('✅ Cloudflared binary ready.')

# Download finetune_worker.py directly from root of repository
!wget -q -O finetune_worker.py https://raw.githubusercontent.com/tushar0067/notebooks/main/finetune_worker.py
print('✅ Fine-tuning worker script downloaded successfully.')

## 🚀 Cell 2 — Start Server
Run each session. Copy the generated `trycloudflare.com` URL into your Web App's Training Page.

In [ ]:
# 🚀 CELL 2 — Start Training Server (Run Each Session)
import threading, uvicorn, subprocess, time, re, sys
sys.path.insert(0, '.')

TUNNEL_URL = None

def start_tunnel():
    global TUNNEL_URL
    p = subprocess.Popen(['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://localhost:8000'],
                         stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    time.sleep(3)
    for line in iter(p.stderr.readline, b''):
        line = line.decode('utf-8')
        if '.trycloudflare.com' in line:
            m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if m:
                TUNNEL_URL = m.group(0)
                print(f'\n\n🔥 COPY THIS URL INTO YOUR TRAINING PAGE:\n\n  {TUNNEL_URL}\n\n')
                break

def run_server():
    import finetune_worker
    uvicorn.run(finetune_worker.app, host='0.0.0.0', port=8000, log_level='warning')

print('🚀 Starting YOLO Fine-Tuning Server...')
threading.Thread(target=start_tunnel, daemon=True).start()
threading.Thread(target=run_server, daemon=True).start()

try:
    while True: time.sleep(1)
except KeyboardInterrupt:
    print('Stopped.')